# Proyecto Oráculo · Notebook de evaluación · Grupo G01

**Análisis y Diseño de Algoritmos · Universidad de Caldas · 2026-2**
Integrante: Jackeline Rivera Aguilera

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J2517/ADA-2026-2-GP01/blob/main/evaluacion_G01.ipynb)

Evalúa la configuración de `entrega.json` sobre un conjunto de prueba **sin editar código, rutas ni parámetros**.

### Cómo ejecutarlo
1. **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU.**
2. **Entorno de ejecución ▸ Ejecutar todo.** La primera celda instala las dependencias y **reinicia la sesión sola**
   (Colab mostrará un aviso de reinicio: es esperado).
3. **Entorno de ejecución ▸ Ejecutar todo** otra vez. Ahora la instalación se salta.
4. Cuando la celda 3 lo pida, **suba el archivo de prueba** (JSON o JSONL). También puede dejarlo antes en
   `/content/` con uno de estos nombres y no se pedirá: `datos_privados.json(l)`, `test_privado.json(l)`, `datos_prueba.json(l)`.

### Qué hace
| Paso | Detalle |
|---|---|
| Configuración | Lee `entrega.json` desde la rama `main` del repositorio y valida índices (0–7) y temperatura (0.0 / 0.3 / 0.7) |
| Datos | Acepta el formato del curso (`prompt`, `ids`, `kwargs`, `id`) y el de IFEval/IFBench (`prompt`, `instruction_id_list`, `kwargs`, `key`) |
| Verificadores | IFEvalG de open-instruct (el del oráculo) + **IFBench** para familias que no están en los datos visibles. Antes de generar, revisa que cada restricción tenga verificador |
| Evaluación | `Oraculo.evaluar` con el modelo del curso (`qwen17b`), `semilla=1`, puntaje todo-o-nada |
| Salida | Precisión, familias que más fallan, `resultados_evaluacion_G01.json` y `.csv` por instancia |

Si se cancela la subida del archivo, corre en **modo demostración** con la partición de validación visible
(sirve para comprobar que todo funciona; **no** es la evaluación oficial).

## 1 · Instalación (se reinicia sola la primera vez) y archivos del repositorio

In [ ]:
import os

MARCA = "/content/.g01_instalado"
if not os.path.exists(MARCA):
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                       nltk spacy emoji langdetect immutabledict syllapy absl-py
    !python -m spacy download en_core_web_sm -q
    # Verificadores: IFEvalG (el que usa oraculo.py) e IFBench (familias fuera de los datos visibles)
    !git clone -q https://github.com/allenai/open-instruct /content/open-instruct
    !git clone -q https://github.com/allenai/IFBench /content/IFBench
    !git -C /content/IFBench checkout -q 1c40f0c10d9b5c5c2f10a175a28007ebb64f7f4d

    # Datos de nltk a mano (el downloader de nltk falla con el proxy de Colab).
    import io, urllib.request, zipfile
    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
    for carpeta, nombre in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"),
                            ("taggers", "averaged_perceptron_tagger"),
                            ("taggers", "averaged_perceptron_tagger_eng"),
                            ("corpora", "stopwords")]:
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

    open(MARCA, "w").write("ok")
    print("Instalación lista. La sesión se reinicia ahora: cuando vuelva, use Ejecutar todo otra vez.")
    os.kill(os.getpid(), 9)   # reinicio obligatorio para que transformers quede bien cargado

# Archivos del grupo, siempre desde la rama main (la versión entregada).
REPO = "https://raw.githubusercontent.com/J2517/ADA-2026-2-GP01/main"
for f in ["oraculo.py", "ayudas.py", "datos_visibles.json", "evaluacion.py", "entrega.json"]:
    !wget -q --no-cache -O {f} {REPO}/{f}
print("Dependencias instaladas y archivos del repositorio descargados.")

## 2 · La configuración entregada

In [ ]:
from evaluacion import cargar_entrega, RANURAS

GRUPO, SEMANA, CONFIG = cargar_entrega("entrega.json")
print(f"Grupo {GRUPO} · semana {SEMANA}")
print("Configuración:", CONFIG)

## 3 · Los datos de prueba

In [ ]:
import hashlib
import os
from evaluacion import leer_instancias

CANDIDATOS = [f"/content/{n}.{e}" for n in ("datos_privados", "test_privado", "datos_prueba")
              for e in ("json", "jsonl")]
RUTA = next((p for p in CANDIDATOS if os.path.exists(p)), None)

if RUTA is None:
    from google.colab import files
    print("Suba el archivo de datos de prueba (JSON o JSONL). Cancelar = modo demostración.")
    try:
        subidos = files.upload()
    except Exception:          # sin interfaz o subida cancelada
        subidos = {}
    RUTA = os.path.abspath(next(iter(subidos))) if subidos else None

if RUTA is None:
    MODO = "demostracion"
    RUTA = "datos_visibles.json"
    INSTANCIAS, aviso = leer_instancias(RUTA)
    INSTANCIAS = [x for x in INSTANCIAS if x.get("split") == "validacion"]
    aviso = "MODO DEMOSTRACIÓN: partición de validación visible (no es la evaluación oficial)"
else:
    MODO = "prueba"
    INSTANCIAS, aviso = leer_instancias(RUTA)

# Caché propio de este archivo: el caché del oráculo usa el id de la instancia como clave,
# así que dos archivos con los mismos ids no deben compartirlo.
HUELLA = hashlib.md5(open(RUTA, "rb").read()).hexdigest()[:10]
print(f"Archivo: {RUTA}  ({len(INSTANCIAS)} instancias, huella {HUELLA})")
if aviso:
    print("Aviso:", aviso)

## 4 · Verificadores y modelo
Primero se revisa, **sin generar texto**, que cada restricción tenga verificador y que sus parámetros sean
válidos. Si alguna instancia no se puede calificar se informa aquí (y se cuenta como fallo en la precisión
sobre el archivo completo), en lugar de detener la evaluación a mitad de camino.

In [ ]:
import os
import sys
from collections import Counter

sys.path.insert(0, "/content/IFBench")
from ifbench import instructions_registry as IFBENCH
from oraculo import Oraculo, _REG as IFEVALG
from ayudas import cargar_modelo
from evaluacion import revisar_verificadores, blindar

REGISTROS = [IFEVALG.INSTRUCTION_DICT, IFBENCH.INSTRUCTION_DICT]   # mismo orden que el oráculo
VALIDAS, PROBLEMAS = revisar_verificadores(INSTANCIAS, REGISTROS)
print(f"Instancias calificables: {len(VALIDAS)} de {len(INSTANCIAS)}")
if PROBLEMAS:
    print("No calificables (familia, motivo):")
    for (fam, motivo), v in Counter((p[1], p[2]) for p in PROBLEMAS).most_common():
        print(f"  {v:3}  {fam}: {motivo}")
print("Familias en el archivo:", len({i for x in INSTANCIAS for i in x["ids"]}))

modelo = cargar_modelo("qwen17b")
oraculo = blindar(Oraculo(modelo, VALIDAS, cache=f"cache_evaluacion_{HUELLA}.json",
                          lote=16, presupuesto=32_000_000,
                          registros=[IFBENCH.INSTRUCTION_DICT]))

## 5 · Evaluación

In [ ]:
import csv, json, time
from evaluacion import resumen

t0 = time.perf_counter()
R = oraculo.evaluar(CONFIG, VALIDAS, semilla=1, desc="evaluando")
SEGUNDOS = time.perf_counter() - t0
RES = resumen(R, VALIDAS)
precision_archivo = RES["aciertos"] / len(INSTANCIAS) if INSTANCIAS else 0.0

print("═" * 60)
print(f" Grupo {GRUPO} · {MODO.upper()}")
print(f" Configuración: {CONFIG}")
print(f" Precisión (instancias calificables): {RES['precision']:.1%}  ({RES['aciertos']}/{RES['n']})")
if PROBLEMAS:
    print(f" Precisión sobre el archivo completo:  {precision_archivo:.1%}  ({RES['aciertos']}/{len(INSTANCIAS)})")
print(f" Tiempo: {SEGUNDOS/60:.1f} min")
print("═" * 60)
print("Familias que más fallan (primera restricción violada):")
for fam, v in RES["familias_que_mas_fallan"]:
    print(f"  {v:3}  {fam}")

salida = {
    "grupo": GRUPO, "semana": SEMANA, "config": CONFIG, "modo": MODO,
    "archivo": os.path.basename(RUTA), "huella_archivo": HUELLA, "modelo": modelo.nombre,
    "semilla": 1, "precision": RES["precision"], "aciertos": RES["aciertos"], "n_calificables": RES["n"],
    "n_archivo": len(INSTANCIAS), "precision_archivo_completo": precision_archivo,
    "no_calificables": [{"id": p[0], "familia": p[1], "motivo": p[2]} for p in PROBLEMAS],
    "familias_que_mas_fallan": RES["familias_que_mas_fallan"], "segundos": round(SEGUNDOS, 1),
}
json.dump(salida, open("resultados_evaluacion_G01.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False)
with open("resultados_evaluacion_G01.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(RES["por_instancia"][0]) if RES["por_instancia"] else ["id"])
    w.writeheader(); w.writerows(RES["por_instancia"])
print("\nGuardado: resultados_evaluacion_G01.json y resultados_evaluacion_G01.csv")

## 6 · Descargar resultados y ver ejemplos de fallos

In [ ]:
for t in R.trazas[:3]:
    print("violó:", t["violo"]); print(t["salida"][:300]); print("-" * 60)

try:
    from google.colab import files
    files.download("resultados_evaluacion_G01.json")
    files.download("resultados_evaluacion_G01.csv")
except Exception:
    pass